# Day 17 — Robust tool calls

Real tools fail: bad arguments, an unknown tool name, an exception, a slow network. A
good agent **never crashes** on a tool failure — it turns the failure into an
**Observation** it can reason about, and **retries** transient errors.

Here you write a `safe_dispatch` that always returns a string, and you wrap a flaky
"search" tool with [`shared.reliability.retry`](../../shared/reliability.py).

> ✅ This notebook runs **offline** — no model needed.

In [2]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

repo root on sys.path: c:\autogen\AILearning


## 🔬 Your turn

Fill in the `TODO`s, then run the cell. Stuck? The solution is below — but try first.

In [ ]:
import random
from shared.tools import calculator, clock
from shared.reliability import retry

TOOLS = {"calculator": calculator, "clock": clock}

# A flaky tool that fails ~50% of the time to simulate a network call.
@retry(attempts=5)
def flaky_search(query):
    if random.random() < 0.5:
        raise ConnectionError("search backend timed out")
    return f"top result for {query!r}: 42"

TOOLS["search"] = flaky_search

def safe_dispatch(action, action_input):
    # Run a tool and ALWAYS return a string observation — never raise.
    # TODO 1: if `action` is not in TOOLS, return a helpful "unknown tool" message
    # TODO 2: otherwise call the tool inside try/except and return "Error: ..." on failure
    raise NotImplementedError

# print(safe_dispatch("calculator", "2+2"))
# print(safe_dispatch("nope", "x"))
# print(safe_dispatch("search", "agents"))

## 🔒 Solution

One correct way to do it. Compare it with your version.

In [3]:
import random
from shared.tools import calculator, clock
from shared.reliability import retry

TOOLS = {"calculator": calculator, "clock": clock}

@retry(attempts=5)
def flaky_search(query):
    if random.random() < 0.5:
        raise ConnectionError("search backend timed out")
    return f"top result for {query!r}: 42"

TOOLS["search"] = flaky_search

def safe_dispatch(action, action_input):
    tool = TOOLS.get(action)
    if tool is None:
        return f"unknown tool: {action!r} (available: {', '.join(TOOLS)})"
    try:
        return str(tool(action_input))
    except Exception as exc:                 # failure becomes a readable observation
        return f"Error running {action}: {exc}"

print(safe_dispatch("calculator", "2+2"))    # 4
print(safe_dispatch("nope", "x"))            # unknown tool
print(safe_dispatch("search", "agents"))     # succeeds thanks to @retry

4
unknown tool: 'nope' (available: calculator, clock, search)
top result for 'agents': 42
